# Visual Diagnostics
Suppose a data scientist has decided to use linear regression to estimate values of one variable (called the response variable) based on another variable (called the predictor). To see how well this method of estimation performs, the data scientist must measure how far off the estimates are from the actual values. These differences are called *residuals*.

$$
\text{residual} ~=~ \text{observed value} ~-~ \text{regression estimate}
$$

A residual is what's left over – the residue – after estimation. 

Residuals are the vertical distances of the points from the regression line. There is one residual for each point in the scatter plot. The residual is the difference between the observed value of $y$ and the fitted value of $y$, so for the point $(x, y)$,

$$
\text{residual} ~~ = ~~ y ~-~
\text{fitted value of }y
~~ = ~~ y ~-~
\text{height of regression line at }x
$$

The function `residual` calculates the residuals. The calculation assumes all the relevant functions we have already defined: `standard_units`, `correlation`, `slope`, `intercept`, and `fit`.

In [1]:
import polars as pl
import numpy as np
from datascience import make_array
import hvplot.polars
import holoviews as hv
from IPython.display import display
path_data = '../../../assets/data/'
from scipy import stats

In [2]:
family_heights = pl.read_csv(path_data + 'family_heights.csv', infer_schema_length=10000)
heights = family_heights.select('midparentHeight', 'childHeight')
heights = heights.rename({'midparentHeight': 'MidParent', 'childHeight': 'Child'})
hybrid = pl.read_csv(path_data + 'hybrid.csv')

In [3]:
def standard_units(x):
    return (x - x.mean())/x.std(ddof=0)

def correlation(table, x, y):
    x_in_standard_units = standard_units(table.get_column(x))
    y_in_standard_units = standard_units(table.get_column(y))
    return (x_in_standard_units * y_in_standard_units).mean()

def slope(table, x, y):
    r = correlation(table, x, y)
    return r * table.get_column(y).std(ddof=0)/table.get_column(x).std(ddof=0)

def intercept(table, x, y):
    a = slope(table, x, y)
    return table.get_column(y).mean() -  a * table.get_column(x).mean()

def fit(table, x, y):
    a = slope(table, x, y)
    b = intercept(table, x, y)
    return a * table.get_column(x) + b

In [4]:
def residual(table, x, y):
    return table.get_column(y) - fit(table, x, y)

Continuing our example of estimating the heights of adult children (the response) based on the midparent height (the predictor), let us calculate the fitted values and the residuals.

In [5]:
heights = heights.with_columns(
        fit(heights, 'MidParent', 'Child').alias('Fitted Value'),
        residual(heights, 'MidParent', 'Child').alias('Residual')
    )
heights

MidParent,Child,Fitted Value,Residual
f64,f64,f64,f64
75.43,73.2,70.712373,2.487627
75.43,69.2,70.712373,-1.512373
75.43,69.0,70.712373,-1.712373
75.43,69.0,70.712373,-1.712373
73.66,73.5,69.584244,3.915756
…,…,…,…
66.64,64.0,65.109971,-1.109971
66.64,62.0,65.109971,-3.109971
66.64,61.0,65.109971,-4.109971


When there are so many variables to work with, it is always helpful to start with visualization. The function `scatter_fit` draws the scatter plot of the data, as well as the regression line. 

In [6]:
def scatter_fit(table, x, y):
    # POLARS_CHANGE: hvplot scatter with a HoloViews Curve overlay for the regression line.
    return (
        table.hvplot.scatter(x=x, y=y, s=15, color="#002fa7")
        * hv.Curve((table.get_column(x).to_numpy(), fit(table, x, y).to_numpy())).opts(color="black", line_width=5)  # dark outline for contrast
        * hv.Curve((table.get_column(x).to_numpy(), fit(table, x, y).to_numpy())).opts(color="#FFFF00", line_width=4)
    ).opts(xlabel=x, ylabel=y)

In [7]:
scatter_fit(heights, 'MidParent', 'Child')

Scatterplot with 'MidParent' on the x-axis and 'Child' on the y-axis. The data points are in dark blue and have a weak to moderate positive association. A yellow line is drawn through the center of the data with a positive slope.

A *residual plot* can be drawn by plotting the residuals against the predictor variable. The function `residual_plot` does just that. 

In [8]:
def residual_plot(table, x, y):
    x_array = table.get_column(x)
    t = pl.DataFrame([
            x_array,
            residual(table, x, y).alias('residuals')
        ])
    xlims = make_array(x_array.min(), x_array.max())
    return (
        t.hvplot.scatter(x=x, y='residuals', color='red')
        * hv.Curve((xlims, make_array(0, 0))).opts(color='darkblue', line_width=4)
    ).opts(title='Residual Plot')

In [9]:
residual_plot(heights, 'MidParent', 'Child')

Scatterplot labeled 'Residual Plot' with 'MidParent' on the x-axis and 'residuals' on the y-axis. The x-axis has the same range as in the previous graph, x=64 to x=74, but the y-axis now ranges from y=-10 to y=10. The data points are in red and have no association; they are a blob. A dark blue horizontal line is drawn at y=0.

The midparent heights are on the horizontal axis, as in the original scatter plot. But now the vertical axis shows the residuals. Notice that the plot appears to be centered around the horizontal line at the level 0 (shown in dark blue). Notice also that the plot shows no upward or downward trend. We will observe later that this lack of trend is true of all regressions.

## Regression Diagnostics
Residual plots help us make visual assessments of the quality of a linear regression analysis. Such assessments are called *diagnostics*. The function ``regression_diagnostic_plots`` draws the original scatter plot as well as the residual plot for ease of comparison.

In [10]:
def regression_diagnostic_plots(table, x, y):
    # POLARS_CHANGE: HoloViews charts are returned, so display shows both figures in order.
    display(scatter_fit(table, x, y))
    display(residual_plot(table, x, y))

In [11]:
regression_diagnostic_plots(heights, 'MidParent', 'Child')

A previous scatterplot is reproduced here without changes: Scatterplot labeled 'Residual Plot' with 'MidParent' on the x-axis and 'residuals' on the y-axis. The x-axis has the same range as in the previous graph, x=64 to x=74, but the y-axis now ranges from y=-10 to y=10. The data points are in red and have no association; they are a blob. A dark blue horizontal line is drawn at y=0.

A previous scatterplot is reproduced here without changes: Scatterplot labeled 'Residual Plot' with 'MidParent' on the x-axis and 'residuals' on the y-axis. The x-axis has the same range as in the previous graph, x=64 to x=74, but the y-axis now ranges from y=-10 to y=10. The data points are in red and have no association; they are a blob. A dark blue horizontal line is drawn at y=0.

This residual plot indicates that linear regression was a reasonable method of estimation. Notice how the residuals are distributed fairly symmetrically above and below the horizontal line at 0, corresponding to the original scatter plot being roughly symmetrical above and below. Notice also that the vertical spread of the plot is fairly even across the most common values of the children's heights. In other words, apart from a few outlying points, the plot isn't narrower in some places and wider in others.

In other words, the accuracy of the regression appears to be about the same across the observed range of the predictor variable. 

**The residual plot of a good regression shows no pattern. The residuals look about the same, above and below the horizontal line at 0, across the range of the predictor variable.**

## Detecting Nonlinearity
Drawing the scatter plot of the data usually gives an indication of whether the relation between the two variables is non-linear. Often, however, it is easier to spot non-linearity in a residual plot than in the original scatter plot. This is usually because of the scales of the two plots: the residual plot allows us to zoom in on the errors and hence makes it easier to spot patterns.

<img src="https://upload.wikimedia.org/wikipedia/commons/7/75/Dugong_dugon.jpg" alt="A dugong; a large marine mammal pictured underwater."/>

Our data are a [dataset](http://www.statsci.org/data/oz/dugongs.html)  on the age and length of dugongs, which are marine mammals related to manatees and sea cows (image from [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Dugong_dugon.jpg)). The data are in a table called `dugong`. Age is measured in years and length in meters. Because dugongs tend not to keep track of their birthdays, ages are estimated based on variables such as the condition of their teeth.

In [12]:
dugong = pl.read_csv(path_data + 'dugongs.csv')
dugong = dugong.select('Length', pl.exclude('Length'))
dugong

Length,Age
f64,f64
1.8,1.0
1.85,1.5
1.87,1.5
1.77,1.5
2.02,2.5
…,…
2.64,16.5
2.56,17.0
2.7,22.5


If we could measure the length of a dugong, what could we say about its age? Let's examine what our data say. Here is a regression of age (the response) on length (the predictor). The correlation between the two variables is substantial, at 0.83.

In [13]:
correlation(dugong, 'Length', 'Age')

0.8296474554905714

High correlation notwithstanding, the plot shows a curved pattern that is much more visible in the residual plot.

In [14]:
regression_diagnostic_plots(dugong, 'Length', 'Age')

Scatterplot titled 'Residual Plot' with 'Length' on the x-axis and 'residuals' on the y-axis. The y-axis ranges from -5 to 15. Data points are in red, and tend to have y values lower than 5, but two data points with high x values have y values greater than 5. A dark blue horizontal line sits above the center of the data at y=0, excluding the outliers.

Scatterplot titled 'Residual Plot' with 'Length' on the x-axis and 'residuals' on the y-axis. The y-axis ranges from -5 to 15. Data points are in red, and tend to have y values lower than 5, but two data points with high x values have y values greater than 5. A dark blue horizontal line sits above the center of the data at y=0, excluding the outliers.

While you can spot the non-linearity in the original scatter, it is more clearly evident in the residual plot.

At the low end of the lengths, the residuals are almost all positive; then they are almost all negative; then positive again at the high end of lengths. In other words the regression estimates have a pattern of being too high, then too low, then too high. That means it would have been better to use a curve instead of a straight line to estimate the ages.

**When a residual plot shows a pattern, there may be a non-linear relation between the variables.**

## Detecting Heteroscedasticity

*Heteroscedasticity* is a word that will surely be of interest to those who are preparing for Spelling Bees. For data scientists, its interest lies in its meaning, which is "uneven spread". 

Recall the table `hybrid` that contains data on hybrid cars in the U.S. Here is a regression of fuel efficiency on the rate of acceleration. The association is negative: cars that accelerate quickly tend to be less efficient.

In [15]:
regression_diagnostic_plots(hybrid, 'acceleration', 'mpg')

Scatterplot titled 'Residual Plot' with 'acceleration' on the x-axis and 'residuals' on the y-axis. A horizontal dark blue line is drawn at y=0. On the left hand side of the plot, the red data points are further away from the dark blue line, both above and below it, than the red data points on the right hand side of the graph.

Scatterplot titled 'Residual Plot' with 'acceleration' on the x-axis and 'residuals' on the y-axis. A horizontal dark blue line is drawn at y=0. On the left hand side of the plot, the red data points are further away from the dark blue line, both above and below it, than the red data points on the right hand side of the graph.

Notice how the residual plot flares out towards the low end of the accelerations. In other words, the variability in the size of the errors is greater for low values of acceleration than for high values. Uneven variation is often more easily noticed in a residual plot than in the original scatter plot.

**If the residual plot shows uneven variation about the horizontal line at 0, the regression estimates are not equally accurate across the range of the predictor variable.**